# 6.3.5 Protocols and Standards

Generate OpenAI-style function call JSON schemas and validate tool calls against them.

In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)

def make_schema(name, desc, params):
    props = {}
    req = []
    for p, m in params.items():
        props[p] = {'type': m['type']}
        if m.get('required', True): req.append(p)
        if 'enum' in m: props[p]['enum'] = m['enum']
    return {'name': name, 'description': desc, 'parameters': {'type': 'object', 'properties': props, 'required': req}}

weather_schema = make_schema('get_weather', 'Get weather', {
    'city': {'type': 'string', 'required': True},
    'unit': {'type': 'string', 'required': False, 'enum': ['celsius', 'fahrenheit']},
})
print(json.dumps(weather_schema, indent=2))

In [ ]:
def validate(schema, args):
    errs = []
    for p in schema['parameters']['required']:
        if p not in args: errs.append(f'Missing: {p}')
    for p, v in args.items():
        props = schema['parameters']['properties']
        if p not in props: errs.append(f'Unknown: {p}')
        elif 'enum' in props[p] and v not in props[p]['enum']:
            errs.append(f'Enum error: {p}={v!r}')
    return errs

tests = [
    ({'city': 'Paris', 'unit': 'celsius'}, 'Valid'),
    ({'unit': 'celsius'}, 'Missing city'),
    ({'city': 'Paris', 'unit': 'kelvin'}, 'Bad enum'),
]
for args, label in tests:
    errs = validate(weather_schema, args)
    print(f'  [{label}] errors={errs}')

valid = [0 if validate(weather_schema, a) else 1 for a, _ in tests]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([l for _, l in tests], valid, color=['mediumseagreen' if v else 'tomato' for v in valid])
ax.set(ylabel='Valid', title='Schema Validation Results'); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('output/protocols_standards.png')
print('Saved protocols_standards.png')